# Graph Algorithms

## Overview
Graph algorithms extract structural insights that are impossible (or extremely expensive) in relational databases: who is the most connected node, what is the shortest path, which nodes are bridges between communities?

**Key algorithms:**

| Algorithm | Question answered | Complexity |
|---|---|---|
| Degree centrality | Who has the most direct connections? | O(V+E) |
| Betweenness centrality | Who is a bridge between groups? | O(V*E) |
| PageRank | Who is most influential (accounting for connection quality)? | O(iterations * E) |
| Shortest path | What is the minimum-hop route between two nodes? | O(V+E) BFS |
| Community detection (Louvain) | What natural clusters exist? | O(E) approx |

**Neo4j GDS (Graph Data Science) library** provides production implementations of these algorithms. The `networkx` demos in this notebook reproduce the same results locally.

---

In [1]:
import networkx as nx

G = nx.MultiDiGraph()
nodes = [
    ("R001","Researcher",{"name":"Aroha Ngata","role":"PI"}),
    ("R002","Researcher",{"name":"Liam Chen","role":"Biostatistician"}),
    ("R003","Researcher",{"name":"Fatima Rashid","role":"Coordinator"}),
    ("R004","Researcher",{"name":"James MacLeod","role":"Researcher"}),
    ("R005","Researcher",{"name":"Priya Sharma","role":"Researcher"}),
    ("I001","Institution",{"name":"Dalhousie University"}),
    ("I002","Institution",{"name":"University of Toronto"}),
    ("I003","Institution",{"name":"McGill University"}),
    ("T001","Trial",{"title":"Cardio Alpha","phase":2}),
    ("T002","Trial",{"title":"Neuro Beta","phase":3}),
    ("T003","Trial",{"title":"Oncology Gamma","phase":1}),
    ("D001","Disease",{"name":"Type 2 Diabetes"}),
    ("D002","Disease",{"name":"Hypertension"}),
    ("D003","Disease",{"name":"Heart Failure"}),
    ("DR001","Drug",{"name":"Metformin"}),
    ("DR002","Drug",{"name":"Lisinopril"}),
    ("DR003","Drug",{"name":"Atorvastatin"}),
]
for nid, label, props in nodes:
    G.add_node(nid, label=label, **props)

edges = [
    ("R001","I001","AFFILIATED_WITH"),("R002","I002","AFFILIATED_WITH"),
    ("R003","I001","AFFILIATED_WITH"),("R004","I003","AFFILIATED_WITH"),
    ("R005","I002","AFFILIATED_WITH"),
    ("R001","T001","LEADS"),("R004","T002","LEADS"),("R005","T003","LEADS"),
    ("R002","T001","CONTRIBUTES_TO"),("R002","T002","CONTRIBUTES_TO"),
    ("R003","T001","CONTRIBUTES_TO"),("R003","T003","CONTRIBUTES_TO"),
    ("T001","D002","STUDIES"),("T001","D001","STUDIES"),
    ("T002","D002","STUDIES"),("T002","D003","STUDIES"),
    ("T003","D001","STUDIES"),
    ("T001","DR002","TESTS"),("T001","DR001","TESTS"),
    ("T002","DR003","TESTS"),("T003","DR001","TESTS"),
    ("DR001","D001","TREATS"),("DR002","D002","TREATS"),
    ("DR003","D002","TREATS"),("DR002","D003","TREATS"),
    ("R001","R004","COLLABORATES_WITH"),("R002","R005","COLLABORATES_WITH"),
    ("R004","R005","COLLABORATES_WITH"),("R001","R003","COLLABORATES_WITH"),
]
for s, d, r in edges:
    G.add_edge(s, d, rel_type=r)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Graph: 17 nodes, 29 edges


---
## Degree centrality

In [2]:
print("=== Degree centrality: who is most connected? ===")
print()

# In-degree, out-degree for all nodes
all_nodes = list(G.nodes(data=True))
rows = []
for nid, data in all_nodes:
    in_deg  = G.in_degree(nid)
    out_deg = G.out_degree(nid)
    rows.append((data.get('label','?'), data.get('name') or data.get('title','?'), in_deg, out_deg))
rows.sort(key=lambda x: -(x[2]+x[3]))

print(f"  {'label':<12s}  {'name':<24s}  {'in':<6s}  {'out':<6s}  total")
print("  " + "-"*60)
for label, name, ind, outd in rows[:12]:
    print(f"  {label:<12s}  {name:<24s}  {ind:<6d}  {outd:<6d}  {ind+outd}")

print()
print("Cypher equivalent (in-degree centrality for Researchers):")
print("""
MATCH (r:Researcher)
WITH  r,
      size([(r)-[]->(x) | x]) AS out_degree,
      size([(x)-[]->(r) | x]) AS in_degree
RETURN r.name, in_degree, out_degree,
       (in_degree + out_degree) AS total_degree
ORDER BY total_degree DESC

-- Neo4j GDS (Graph Data Science library):
CALL gds.degree.stream('myGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score AS degree
ORDER BY degree DESC
""")


=== Degree centrality: who is most connected? ===

  label         name                      in      out     total
  ------------------------------------------------------------
  Trial         Cardio Alpha              3       4       7
  Trial         Neuro Beta                2       3       5
  Researcher    Aroha Ngata               0       4       4
  Researcher    Liam Chen                 0       4       4
  Researcher    Fatima Rashid             1       3       4
  Researcher    James MacLeod             1       3       4
  Researcher    Priya Sharma              2       2       4
  Trial         Oncology Gamma            2       2       4
  Disease       Hypertension              4       0       4
  Disease       Type 2 Diabetes           3       0       3
  Drug          Metformin                 2       1       3
  Drug          Lisinopril                1       2       3

Cypher equivalent (in-degree centrality for Researchers):

MATCH (r:Researcher)
WITH  r,
      size([

---
## Shortest path

In [3]:
import networkx as nx

print("=== Shortest path between researchers ===")
print()

# Build undirected version for path finding
G_undir = G.to_undirected()

# Find shortest paths between all researcher pairs
researchers = [n for n, d in G.nodes(data=True) if d.get('label') == 'Researcher']

print("Shortest paths between researcher pairs (hop count):")
print(f"  {'From':<20s}  {'To':<20s}  {'Hops':<6s}  Path")
print("  " + "-"*75)
for i, r1 in enumerate(researchers):
    for r2 in researchers[i+1:]:
        try:
            path = nx.shortest_path(G_undir, r1, r2)
            path_names = [G.nodes[n].get('name') or G.nodes[n].get('title','?') for n in path]
            hops = len(path) - 1
            print(f"  {G.nodes[r1]['name']:<20s}  {G.nodes[r2]['name']:<20s}  {hops:<6d}  {' -> '.join(path_names)}")
        except nx.NetworkXNoPath:
            print(f"  {G.nodes[r1]['name']:<20s}  {G.nodes[r2]['name']:<20s}  no path")

print()
print("Cypher shortest path:")
print("""
-- Shortest path (any relationship type):
MATCH p = shortestPath(
    (r1:Researcher {name: 'Aroha Ngata'})-[*]-(r2:Researcher {name: 'Priya Sharma'})
)
RETURN p, length(p) AS hops,
       [n IN nodes(p) | coalesce(n.name, n.title)] AS path_names

-- Shortest path via specific relationship types:
MATCH p = shortestPath(
    (r1:Researcher)-[:COLLABORATES_WITH|CONTRIBUTES_TO*]-(r2:Researcher)
)
WHERE r1.name = 'Aroha Ngata' AND r2.name = 'Priya Sharma'
RETURN length(p) AS hops
""")


=== Shortest path between researchers ===

Shortest paths between researcher pairs (hop count):
  From                  To                    Hops    Path
  ---------------------------------------------------------------------------
  Aroha Ngata           Liam Chen             2       Aroha Ngata -> Cardio Alpha -> Liam Chen
  Aroha Ngata           Fatima Rashid         1       Aroha Ngata -> Fatima Rashid
  Aroha Ngata           James MacLeod         1       Aroha Ngata -> James MacLeod
  Aroha Ngata           Priya Sharma          2       Aroha Ngata -> James MacLeod -> Priya Sharma
  Liam Chen             Fatima Rashid         2       Liam Chen -> Cardio Alpha -> Fatima Rashid
  Liam Chen             James MacLeod         2       Liam Chen -> Neuro Beta -> James MacLeod
  Liam Chen             Priya Sharma          1       Liam Chen -> Priya Sharma
  Fatima Rashid         James MacLeod         2       Fatima Rashid -> Aroha Ngata -> James MacLeod
  Fatima Rashid         Priya Sharm

---
## PageRank and betweenness centrality

In [4]:
import networkx as nx

print("=== PageRank and betweenness centrality ===")
print()

# Convert to simple DiGraph for algorithms
G_simple = nx.DiGraph()
for n, d in G.nodes(data=True):
    G_simple.add_node(n, **d)
for u, v, d in G.edges(data=True):
    if not G_simple.has_edge(u, v):
        G_simple.add_edge(u, v)

# PageRank
pr = nx.pagerank(G_simple, alpha=0.85)
pr_sorted = sorted(pr.items(), key=lambda x: -x[1])

print("PageRank (top 10 most 'important' nodes):")
print(f"  {'label':<14s}  {'name':<24s}  score")
print("  " + "-"*50)
for nid, score in pr_sorted[:10]:
    data  = G.nodes[nid]
    label = data.get('label','?')
    name  = data.get('name') or data.get('title','?')
    print(f"  {label:<14s}  {name:<24s}  {score:.4f}")

print()
# Betweenness centrality (undirected)
G_undir_simple = G_simple.to_undirected()
bc = nx.betweenness_centrality(G_undir_simple)
bc_sorted = sorted(bc.items(), key=lambda x: -x[1])
print("Betweenness centrality (top 8 bridge nodes):")
print(f"  {'label':<14s}  {'name':<24s}  score")
print("  " + "-"*50)
for nid, score in bc_sorted[:8]:
    data  = G.nodes[nid]
    label = data.get('label','?')
    name  = data.get('name') or data.get('title','?')
    print(f"  {label:<14s}  {name:<24s}  {score:.4f}")

print()
print("Neo4j GDS equivalents:")
print("""
-- PageRank:
CALL gds.pageRank.stream('myGraph', {dampingFactor: 0.85, maxIterations: 20})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score
ORDER BY score DESC LIMIT 10

-- Betweenness centrality:
CALL gds.betweenness.stream('myGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score AS betweenness
ORDER BY betweenness DESC LIMIT 10

-- Community detection (Louvain):
CALL gds.louvain.stream('myGraph')
YIELD nodeId, communityId
RETURN gds.util.asNode(nodeId).name AS name, communityId
ORDER BY communityId
""")


=== PageRank and betweenness centrality ===

PageRank (top 10 most 'important' nodes):
  label           name                      score
  --------------------------------------------------
  Disease         Type 2 Diabetes           0.1334
  Disease         Hypertension              0.1173
  Drug            Metformin                 0.0721
  Disease         Heart Failure             0.0655
  Trial           Oncology Gamma            0.0649
  Institution     University of Toronto     0.0606
  Trial           Cardio Alpha              0.0572
  Researcher      Priya Sharma              0.0504
  Institution     Dalhousie University      0.0504
  Trial           Neuro Beta                0.0504

Betweenness centrality (top 8 bridge nodes):
  label           name                      score
  --------------------------------------------------
  Trial           Cardio Alpha              0.3586
  Researcher      James MacLeod             0.1999
  Trial           Neuro Beta                0.166

---
## Common Pitfalls

**1. Running graph algorithms without creating a named graph projection**
Neo4j GDS algorithms require a graph projection (`gds.graph.project`) before running. Running an algorithm directly on the native graph without a projection fails or produces incorrect results. Always `CALL gds.graph.project(...)` first.

**2. Treating PageRank scores as absolute importance values**
PageRank scores are relative within the graph and depend on the damping factor and number of iterations. A score of 0.02 in one graph is meaningless compared to 0.02 in another. Use scores for relative ranking within the same graph only.

**3. Using betweenness centrality on disconnected graphs**
Betweenness centrality is undefined or 0 for nodes in components that have no path between them. Run `gds.wcc.stream` (Weakly Connected Components) first to understand graph connectivity before running centrality algorithms.

**4. Ignoring edge direction in undirected algorithms**
Some GDS algorithms treat graphs as undirected by default. If your relationships are directionally meaningful (LEADS, REPORTS_TO), use the appropriate orientation: `NATURAL`, `REVERSE`, or `UNDIRECTED` in the graph projection.

**5. Running algorithms on the full graph during production queries**
Betweenness centrality is O(V*E) — expensive on large graphs. Pre-compute algorithm results and store them as node properties: `CALL gds.pageRank.write('myGraph', {writeProperty: 'pagerank_score'})`. Query the stored score instead of recomputing each time.

**6. Not filtering graph projections**
Including all node labels and relationship types in a projection when an algorithm only needs a subset wastes memory and compute. Project only the relevant subgraph: `gds.graph.project('collab', 'Researcher', 'COLLABORATES_WITH')`.


---
*sql_methods_library - Samantha McGarrigle*